🎧 **Audio introduction**

> *New to this notebook? The audio guide before you start.*




<audio controls>
  <source src="https://raw.githubusercontent.com/fair-vid/video-multimodal-pipeline/main/assets/audio/audio_tesseract-ocr_notebook_tutorial.wav" type="audio/wav">
</audio>

[📌 Google Drive data structure info](https://github.com/fair-vid/video-multimodal-pipeline/blob/main/notebooks/add_doc_dir_struct/README.md)


---

# 📄 Document OCR Agent — Image Text Extraction with Layout Metadata

**Paper about ** *an Agent-Based Digital Twin for AI in Student Admissions*
**Notebook role:** OCR-based document preprocessing agent
**Demo stage:** Document understanding (text + spatial layout)

---

## Purpose of This Notebook

This notebook implements the **OCR-based document processing agent** used in the admissions process digital twin described in the paper.
It processes scanned or photographed applicant documents (e.g., diplomas, certificates, transcripts) and extracts:

* raw textual content using Tesseract OCR,
* word-level bounding boxes and spatial layout metadata,
* preprocessed images optimized for noisy, low-quality scans.

The extracted outputs serve as **human-readable and machine-interpretable artifacts** that are later consumed by higher-level reasoning agents.

---

## Role in the Digital Twin Architecture

In the full system, this OCR agent reacts automatically to document upload events and runs **in parallel** with vision-based zero-shot document understanding agents.

For the Google Colab demonstration, the workflow is **flattened into sequential notebooks**, where each notebook corresponds to one agent or processing step.

This notebook produces:

* `documents_image_text_pytesseract/` — extracted text files
* `documents_image_text_pytesseract_box_info/` — word-level bounding boxes (TSV)

These outputs are later fused with:

* zero-shot vision–language model outputs,
* holistic document reasoning agents,
* administrator-facing inspection tools.

---

## Execution Notes

* The notebook is designed to run **without user interaction** once documents are present.
* You may safely use **Runtime → Run all**.
* All outputs are stored locally in your Google Drive under:

  ```
  colab-output/dream_applicant_application/
  ```

---

## Demo Context & Availability

This notebook is part of a **reproducible demo pipeline** accompanying the paper submission.

> **Note:**
> A public GitHub repository containing all notebooks, prompts, and orchestration code will be released after the anonymous review process is completed.

---


In [ ]:
import os
from pathlib import Path

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
    print("Google Drive mounted")
else:
    print("Google Drive already mounted")

Google Drive already mounted


In [ ]:
#@title # ✔ Check if we can find at least one image in any of subdirs ../documents_image/

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"
os.makedirs(drive_dir, exist_ok=True)

subdirs = [d for d in os.listdir(drive_dir) if os.path.isdir(os.path.join(drive_dir, d))]
print(subdirs)

base_path = "/content/drive/MyDrive/colab-output/dream_applicant_application/"
target_demo_dir = os.path.join(base_path, "0_0/documents_image/")


has_any_image = False

if subdirs:
    for sd in subdirs:
        img_path = Path(base_path) / sd / "documents_image"
        if img_path.exists() and img_path.is_dir():
            # Check if there is at least one file in this directory
            files = [f for f in img_path.iterdir() if f.is_file()]
            if len(files) > 0:
                has_any_image = True
                break

if not subdirs or not has_any_image:
    print("⚠️ ATTENTION: No document images found or directories are empty!")
    print("To test this OCR framework, you need to provide at least one document image.")

    # Create the specific directory for the user
    os.makedirs(target_demo_dir, exist_ok=True)

    print(f"\n📁 Created directory: {target_demo_dir}")
    print("👉 ACTION REQUIRED: Please upload at least one image file (JPG, PNG) into the folder above and re-run the processing cells below.")
else:
    print(f"✅ Verified: Found valid document images within the subdirectories. Proceed to run the collection and processing cells.")

['0_0', '18190_27580', '40529_33700', '60811_46042']
✅ Verified: Found valid document images within the subdirectories. Proceed to run the collection and processing cells.


In [ ]:


drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"

# Dictionary to store files by subdirectory
files_by_subdir = {}

# Iterate through direct subdirectories
for subdir in Path(drive_dir).iterdir():
    if subdir.is_dir():
        # Check if 'documents_image' exists in this subdirectory
        documents_image_path = subdir / "documents_image"

        if documents_image_path.exists() and documents_image_path.is_dir():
            # Get all direct files (not subdirectories) in documents_image
            files = [f for f in documents_image_path.iterdir() if f.is_file()]
            files_by_subdir[subdir.name] = files

            print(f"Found {len(files)} files in {subdir.name}/documents_image/")

'''# Now you can use the files later
for subdir_name, files in files_by_subdir.items():
    print(f"\n{subdir_name}:")
    for file in files:
        print(f"  - {file.name}")'''

Found 12 files in 18190_27580/documents_image/
Found 12 files in 40529_33700/documents_image/
Found 5 files in 60811_46042/documents_image/


'# Now you can use the files later\nfor subdir_name, files in files_by_subdir.items():\n    print(f"\n{subdir_name}:")\n    for file in files:\n        print(f"  - {file.name}")'

In [ ]:
# Install Tesseract and Python wrappers
!apt update -qq
!apt install -y tesseract-ocr tesseract-ocr-eng -qq
!pip install --quiet pytesseract pillow opencv-python-headless

import pytesseract
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Tesseract version:", pytesseract.get_tesseract_version())

28 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 28 not upgraded.
Tesseract version: 4.1.1


In [ ]:
#!pip install opencv-python

In [ ]:
#@title preprocess image functions
import cv2
import numpy as np
from PIL import Image

def preprocess_for_forms(image_path):
    '''
    # 1. Read the image
    img = cv2.imread(str(image_path))

    # 2. Convert to Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    '''


    # 1. SAFELY read the image (handles spaces, parentheses, and path issues)

    # Get the absolute, full path to the file to guarantee we find it
    safe_path = str(Path(image_path).resolve())

    # Read the file as raw data first to bypass OpenCV's path reading bugs
    img_array = np.fromfile(safe_path, np.uint8)

    # If the array is empty, the file path is wrong or the file doesn't exist
    if len(img_array) == 0:
        raise ValueError(f"FILE NOT FOUND OR EMPTY: {safe_path}")

    # Decode the raw data into an image
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    # If it's still None, the file is corrupted or not a real JPEG
    if img is None:
        raise ValueError(f"NOT A VALID IMAGE FORMAT: {safe_path}")

    # 2. Convert to Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)








    # 3. OPTIONAL: Morphological opening to remove thin watermarks
    # This erodes the image (eats away thin lines) then dilates it (restores thickness).
    # Since the "IELTS" watermark is thin and the real text is bold, this helps remove the noise.
    kernel = np.ones((2,2), np.uint8)
    gray = cv2.morphologyEx(gray, cv2.MORPH_OPEN, kernel)

    # 4. Apply Gaussian Adaptive Thresholding
    # blockSize (31): Checks a 31x31 pixel area.
    # C (25): This is the aggression level.
    #         Higher C = Removes more grey background/watermarks, but might break thin text.
    #         Lower C = Keeps more background noise.
    #         For this specific image, a higher C (20-25) helps kill the grey boxes.
    processed = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        25
    )

    # 5. Denoise slightly to clean up speckles
    # This removes small black dots that might remain after thresholding
    processed = cv2.fastNlMeansDenoising(processed, None, 10, 7, 21)

    return Image.fromarray(processed)

def preprocess_image(image_path):
    """
    Reads an image path, converts to grayscale, applies adaptive thresholding
    to remove dark/grey backgrounds, and ensures black text on white background.
    Returns a PIL Image object.
    """
    # 1. Read image using OpenCV (convert Path to str just in case)
    # cv2.imread loads the image as a numpy array
    img_cv = cv2.imread(str(image_path))

    # Check if image loaded successfully
    if img_cv is None:
        raise ValueError(f"OpenCV could not open: {image_path}")

    # 2. Convert to Grayscale
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)

    # 3. Apply Adaptive Thresholding
    # This is the key step for dark/grey backgrounds.
    # It calculates the threshold for small regions (blockSize) individually.
    # blockSize (31): Size of a pixel neighborhood that is used to calculate a threshold value.
    #                 Must be an odd number (e.g., 11, 31, 51).
    # C (15): Constant subtracted from the mean. Higher = less noise, but might remove thin text.
    processed_img = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        15
    )

    # 4. Check for "Inverted" Text (White text on Black background)
    # Tesseract prefers Black text on White background.
    # We count the non-zero (white) pixels.
    # If the image is mostly Black (0), it means the text is White. We need to invert it.
    white_pixels = np.count_nonzero(processed_img)
    total_pixels = processed_img.size

    # If less than 50% of the image is white, it's likely a dark background with light text
    # so we invert it to make the background white.
    if white_pixels < (total_pixels * 0.5):
        processed_img = cv2.bitwise_not(processed_img)

    # 5. Denoise (Optional) - removing salt-and-pepper noise
    # processed_img = cv2.medianBlur(processed_img, 3)

    # 6. Convert back to PIL Image
    # This allows it to be a direct drop-in replacement for Image.open()
    return Image.fromarray(processed_img)

In [ ]:
import os
from pathlib import Path
from PIL import Image
import pytesseract
import cv2

# Iterate through each subdirectory in files_by_subdir
for subdir_name, files in files_by_subdir.items():
    # Create the output directory for text files
    subdir_path = Path(drive_dir) / subdir_name
    output_dir = subdir_path / "documents_image_text_pytesseract"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nProcessing {subdir_name}...")

    # Process each image file
    for image_file in files:
        # Create output text file path with same name (but .txt extension)
        output_file = output_dir / f"{image_file.stem}.txt"


        if output_file.exists() :  # Skip if output file already exists    and '28494_IELTS' not in str(image_file)
            print(f"\r  ⊘ Skipped (already exists): {image_file.name[:2]}", end='')
            continue

        try:
            #img = Image.open(image_file)
            #img = preprocess_image(image_file) # no improve
            img = preprocess_for_forms(image_file)
            #img.save( f"proc_{image_file.stem}.png")

            custom_config = r'--oem 1 --psm 6'  # --psm 6: Assume a single uniform block of text. (Great for aligned rows)  # --psm 4: Assume a single column of variable sizes.  # --oem 1: Neural Net LSTM engine only (usually most accurate)
            #text = pytesseract.image_to_string(img, lang='eng')
            text = pytesseract.image_to_string(img, lang='eng', config=custom_config)

            # Save extracted text
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(text)

            #print(f"  ✓ Processed: {image_file.name} -> {output_file.name}")

        except Exception as e:
            print(f"  ✗ Error processing {image_file.name}: {str(e)}")

print("\nAll images processed!")


Processing 18190_27580...
  ⊘ Skipped (already exists): pa
Processing 40529_33700...

Processing 60811_46042...

All images processed!


In [ ]:
import os
from pathlib import Path
from PIL import Image
import pytesseract

# Iterate through each subdirectory in files_by_subdir
for subdir_name, files in files_by_subdir.items():
    # Create the output directory for box info files
    subdir_path = Path(drive_dir) / subdir_name
    output_dir = subdir_path / "documents_image_text_pytesseract_box_info"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nProcessing {subdir_name} for box info...")

    # Process each image file
    for image_file in files:
        # Create output TSV file path with same name (but .tsv extension)
        output_file = output_dir / f"{image_file.stem}.tsv"

        # Skip if output file already exists
        if output_file.exists():
            #print(f"  ⊘ Skipped (already exists): {image_file.name}")
            continue

        try:
            # Open image and extract text with bounding boxes
            #img = Image.open(image_file)
            img = preprocess_for_forms(image_file)

            data = pytesseract.image_to_data(img, lang='eng', output_type=pytesseract.Output.DICT)

            # Save box info to TSV file
            with open(output_file, 'w', encoding='utf-8') as f:
                # Write header
                f.write("word\tx\ty\twidth\theight\n")

                # Write each word with its bounding box
                for i, word in enumerate(data['text']):
                    if word.strip():
                        x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
                        f.write(f"{word}\t{x}\t{y}\t{w}\t{h}\n")

            #print(f"  ✓ Processed: {image_file.name} -> {output_file.name}")

        except Exception as e:
            print(f"  ✗ Error processing {image_file.name}: {str(e)}")

print("\nAll box info extracted!")


Processing 18190_27580 for box info...

Processing 40529_33700 for box info...

Processing 60811_46042 for box info...

All box info extracted!




---

## ▶️ Next Step: Vision Language Document Understanding (Local LLM)

In the full admissions **process digital twin**, document analysis is performed by **multiple AI agents in parallel**.
While this notebook extracted **text and spatial layout** using OCR, a complementary agent performs **semantic interpretation directly from document images** using a vision-capable large language model.

For demonstration purposes in Google Colab, these agents are executed **sequentially as separate notebooks**, even though they operate in parallel in the real system.

### What happens next

In the next notebook, document images are processed using a **local vision language model (LMM)** running via **Ollama**.
This agent:

* interprets document structure and content in zero-shot mode,
* infers document type and semantic sections,
* produces machine-readable descriptions (e.g., JSON-like outputs),
* complements OCR results with higher-level reasoning.

The OCR outputs generated here are later **reconciled** with the LLM outputs to support transparency, verification, and human review.

### Open the next notebook here:

**Vision Language Document Understanding (Ollama, Local LLM)**
[https://colab.research.google.com/drive/12eEfiEB9rIf1RZOIISw-uE_b_mE42xBL?usp=sharing](https://colab.research.google.com/drive/12eEfiEB9rIf1RZOIISw-uE_b_mE42xBL?usp=sharing)

> **Note:**
> In the research system described in the paper, OCR-based extraction and vision-language reasoning start at the same time after document upload.
> The sequential structure used here is only for clarity and reproducibility in Colab.

---
